# 03.1 Data Augmentation (v2)

**Purpose**: Recompute derived and rolling features with correct grouping, ordering, and season resets.

**Input**: `data/processed/master_races_with_2025.csv`

**Output**: `data/processed/master_races_augmented_with_2025.csv`


In [45]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
PROCESSED_ROOT = Path("data/processed")
INPUT_PATH = PROCESSED_ROOT / "master_races_with_2025.csv"
OUTPUT_PATH = PROCESSED_ROOT / "master_races_augmented.csv"

print(f"Loading: {INPUT_PATH}")
master = pd.read_csv(INPUT_PATH, low_memory=False)

# Normalize date and sort for correct temporal ordering
master['date'] = pd.to_datetime(master['date'], errors='coerce')
master = master.sort_values(['year', 'round', 'date']).reset_index(drop=True)

# Ensure numeric fields used in calculations
for col in ['points', 'grid', 'positionOrder', 'position', 'podium']:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors='coerce')

print(f"Rows: {len(master):,}")
print(f"Years: {master['year'].min()} - {master['year'].max()}")


Loading: data\processed\master_races_with_2025.csv
Rows: 12,817
Years: 1994 - 2025


## 1. Standings Points (Season-Reset) + Positions

Preserve official standings values when present, and only fill missing standings from season-reset race-point calculations. Then derive positions only where missing.

In [46]:
# Driver standings points (season reset)
driver_points_calc = (
    master.groupby(['year', 'driverId'])['points']
    .cumsum()
)

# Fill only missing driver standings points
if 'driver_standings_points' not in master.columns:
    master['driver_standings_points'] = driver_points_calc
else:
    master['driver_standings_points'] = master['driver_standings_points'].fillna(driver_points_calc)

# Driver standings position (rank within each race)
driver_pos_calc = (
    master.groupby(['year', 'round'])['driver_standings_points']
    .rank(method='min', ascending=False)
)

if 'driver_standings_position' not in master.columns:
    master['driver_standings_position'] = driver_pos_calc
else:
    master['driver_standings_position'] = master['driver_standings_position'].fillna(driver_pos_calc)

# ----------------------------------------------------------------------
# Constructor standings points (season reset, monotonic)
# ----------------------------------------------------------------------
# Use computed race-point cumsum only as fallback for missing values,
# preserving official standings values (including deductions) when present.
constructor_race_points = (
    master.groupby(['year', 'round', 'constructorId'], as_index=False)['points']
    .sum()
    .rename(columns={'points': 'constructor_race_points'})
)

constructor_standings = constructor_race_points.sort_values(['year', 'constructorId', 'round']).copy()
constructor_standings['constructor_standings_points_calc'] = (
    constructor_standings.groupby(['year', 'constructorId'])['constructor_race_points']
    .cumsum()
)

master = master.merge(
    constructor_standings[['year', 'round', 'constructorId', 'constructor_standings_points_calc']],
    on=['year', 'round', 'constructorId'],
    how='left'
)

if 'constructor_standings_points' not in master.columns:
    master['constructor_standings_points'] = master['constructor_standings_points_calc']
else:
    master['constructor_standings_points'] = master['constructor_standings_points'].fillna(master['constructor_standings_points_calc'])

master = master.drop(columns=['constructor_standings_points_calc'])

# Constructor standings position (rank within each race)
constructor_pos_calc = (
    master.groupby(['year', 'round'])['constructor_standings_points']
    .rank(method='min', ascending=False)
)
if 'constructor_standings_position' not in master.columns:
    master['constructor_standings_position'] = constructor_pos_calc
else:
    master['constructor_standings_position'] = master['constructor_standings_position'].fillna(constructor_pos_calc)


## 2. Pre‑Race Standings (Shifted)

Create pre‑race standings by shifting within season. Leave season start as NaN; fill any other missing positions by ranking pre‑race points.

In [47]:
# Pre-race points (shift within season)
master['driver_standings_points_PRE_RACE'] = (
    master.groupby(['year', 'driverId'])['driver_standings_points']
    .shift(1)
)

# Constructor pre-race points must shift by constructor-race, not by driver row.
# This ensures both constructor rows in a race get the same prior-race value.
constructor_round = (
    master[['year', 'round', 'constructorId', 'constructor_standings_points']]
    .drop_duplicates(subset=['year', 'round', 'constructorId'])
    .sort_values(['year', 'constructorId', 'round'])
)
constructor_round['constructor_standings_points_PRE_RACE'] = (
    constructor_round.groupby(['year', 'constructorId'])['constructor_standings_points']
    .shift(1)
)

master = master.drop(columns=['constructor_standings_points_PRE_RACE'], errors='ignore')
master = master.merge(
    constructor_round[['year', 'round', 'constructorId', 'constructor_standings_points_PRE_RACE']],
    on=['year', 'round', 'constructorId'],
    how='left'
)

# Pre-race positions: rank pre-race points within each race
master['driver_standings_position_PRE_RACE'] = (
    master.groupby(['year', 'round'])['driver_standings_points_PRE_RACE']
    .rank(method='min', ascending=False)
)
master['constructor_standings_position_PRE_RACE'] = (
    master.groupby(['year', 'round'])['constructor_standings_points_PRE_RACE']
    .rank(method='min', ascending=False)
)

# Leave season-start rows as NaN
first_round_by_driver = master.groupby(['year', 'driverId'])['round'].transform('min')
first_round_by_constructor = master.groupby(['year', 'constructorId'])['round'].transform('min')

is_driver_season_start = master['round'] == first_round_by_driver
is_constructor_season_start = master['round'] == first_round_by_constructor

master.loc[is_driver_season_start, 'driver_standings_position_PRE_RACE'] = np.nan
master.loc[is_constructor_season_start, 'constructor_standings_position_PRE_RACE'] = np.nan

print("✓ Pre-race standings created (season-start left NaN)")
print(f"  driver_standings_position_PRE_RACE missing: {master['driver_standings_position_PRE_RACE'].isna().sum():,}")
print(f"  constructor_standings_position_PRE_RACE missing: {master['constructor_standings_position_PRE_RACE'].isna().sum():,}")


✓ Pre-race standings created (season-start left NaN)
  driver_standings_position_PRE_RACE missing: 799
  constructor_standings_position_PRE_RACE missing: 696


## 3. Rolling Averages (Points/Grid/Position)

Grouped rolling features with proper shift to avoid leakage.

In [48]:
def rolling_mean_by_group(df, group_col, value_col, window, col_name):
    df[col_name] = (
        df.groupby(group_col, group_keys=False)[value_col]
        .apply(lambda s: s.shift(1).rolling(window=window, min_periods=1).mean())
    )

# Driver rolling averages
if 'points' in master.columns:
    rolling_mean_by_group(master, 'driverId', 'points', 10, 'driver_points_avg_last_10')
if 'grid' in master.columns:
    rolling_mean_by_group(master, 'driverId', 'grid', 5, 'driver_avg_grid_last_5')
if 'positionOrder' in master.columns:
    rolling_mean_by_group(master, 'driverId', 'positionOrder', 5, 'driver_avg_position_last_5')

print("✓ Rolling averages created")


✓ Rolling averages created


## 4. Career Totals (Podiums + Races Completed)

Cumulative features grouped per driver (no cross-driver bleed).

In [49]:
# Driver total podiums (grouped cumulative, shifted)
if 'podium' in master.columns:
    master['driver_total_podiums'] = (
        master.groupby('driverId', group_keys=False)['podium']
        .apply(lambda s: s.shift(1).fillna(0).cumsum())
    )
else:
    master['driver_total_podiums'] = np.nan

# Driver races completed (previous races count)
master['driver_races_completed'] = master.groupby('driverId').cumcount()

print("✓ Career totals created")
print(f"  driver_total_podiums max: {master['driver_total_podiums'].max():.0f}")
print(f"  driver_races_completed max: {master['driver_races_completed'].max():.0f}")


✓ Career totals created
  driver_total_podiums max: 202
  driver_races_completed max: 426


## 5. Status Category + Rolling Status Rates

Compute simplified status categories and rolling rates per driver.

In [50]:
def categorize_status(statusId):
    if pd.isna(statusId):
        return "Unknown"
    try:
        statusId = int(statusId)
    except (ValueError, TypeError):
        return "Unknown"

    if statusId == 1:
        return "Finished"
    elif statusId in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 45, 50, 53, 55, 58, 88,
                      111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 122, 123, 124, 125, 127, 133, 134]:
        return "Finished_Lapped"
    elif statusId == 2:
        return "Disqualified"
    elif statusId == 62:
        return "Not_Classified"
    else:
        return "DNF"

master['status_category'] = master['statusId'].apply(categorize_status)

status_categories = [
    "Finished_Lapped",
    "DNF",
    "Disqualified",
    "Not_Classified",
]

for category in status_categories:
    temp_col = f"_temp_{category.lower()}"
    master[temp_col] = (master['status_category'] == category).astype(int)
    master[f"{category.lower()}_rate_last_10"] = (
        master.groupby('driverId', group_keys=False)[temp_col]
        .apply(lambda s: s.shift(1).rolling(window=10, min_periods=1).mean())
    )
    master.drop(columns=[temp_col], inplace=True)

print("✓ Status categories and rolling rates created")


✓ Status categories and rolling rates created


## 6. Podium Rates + Sprint Points Averages

Driver podium rates (rolling) and sprint points averages.

In [51]:
if 'podium' in master.columns:
    for window in [3, 5, 10]:
        master[f'driver_podium_rate_last_{window}'] = (
            master.groupby('driverId', group_keys=False)['podium']
            .apply(lambda s: s.shift(1).rolling(window=window, min_periods=1).mean())
        )

# Constructor podium rate last 15 (race-level; shift excludes current race)
# NOTE: master is per driver row, so we must compute podium-per-constructor-per-race first
# to avoid mixing teammate rows from the same race.
if 'podium' in master.columns and 'raceId' in master.columns:
    constructor_race = (
        master.groupby(['year', 'round', 'raceId', 'constructorId'], as_index=False)['podium']
        .max()
        .rename(columns={'podium': 'constructor_podium_race'})
    )
    constructor_race = constructor_race.sort_values(['year', 'round', 'constructorId', 'raceId'])

    constructor_race['constructor_podium_rate_last_15'] = (
        constructor_race.groupby('constructorId', group_keys=False)['constructor_podium_race']
        .apply(lambda s: s.shift(1).rolling(window=15, min_periods=1).mean())
    )

    master = master.drop(columns=['constructor_podium_rate_last_15'], errors='ignore')
    master = master.merge(
        constructor_race[['year', 'round', 'raceId', 'constructorId', 'constructor_podium_rate_last_15']],
        on=['year', 'round', 'raceId', 'constructorId'],
        how='left'
    )
else:
    master['constructor_podium_rate_last_15'] = np.nan

# Sprint points rolling averages (if sprint results exist)
if 'sprint_results_points' in master.columns:
    master['sprint_results_points'] = pd.to_numeric(master['sprint_results_points'], errors='coerce')
    master['driver_sprint_points_avg_last_5'] = (
        master.groupby('driverId', group_keys=False)['sprint_results_points']
        .apply(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
    )

print("✓ Podium rates and sprint points averages created")


✓ Podium rates and sprint points averages created


## 7. Circuit-Specific Features

Experience and performance at each circuit.

In [52]:
# Driver races at circuit (count of previous races at this circuit)
if 'circuitId' in master.columns:
    master['driver_races_at_circuit'] = (
        master.groupby(['driverId', 'circuitId'])
        .cumcount()
    )

# Driver podium rate at circuit
if 'podium' in master.columns and 'circuitId' in master.columns:
    master['driver_podium_rate_at_circuit'] = (
        master.groupby(['driverId', 'circuitId'], group_keys=False)['podium']
        .apply(lambda s: s.shift(1).expanding(min_periods=1).mean())
    )

# Constructor podium rate at circuit (race-level; shift excludes current race)
if 'podium' in master.columns and 'circuitId' in master.columns and 'raceId' in master.columns:
    constructor_circuit_race = (
        master.groupby(['year', 'round', 'raceId', 'constructorId', 'circuitId'], as_index=False)['podium']
        .max()
        .rename(columns={'podium': 'constructor_podium_race'})
    )
    constructor_circuit_race = constructor_circuit_race.sort_values(['constructorId', 'circuitId', 'year', 'round', 'raceId'])

    constructor_circuit_race['constructor_podium_rate_at_circuit'] = (
        constructor_circuit_race.groupby(['constructorId', 'circuitId'], group_keys=False)['constructor_podium_race']
        .apply(lambda s: s.shift(1).expanding(min_periods=1).mean())
    )

    master = master.drop(columns=['constructor_podium_rate_at_circuit'], errors='ignore')
    master = master.merge(
        constructor_circuit_race[['year', 'round', 'raceId', 'constructorId', 'circuitId', 'constructor_podium_rate_at_circuit']],
        on=['year', 'round', 'raceId', 'constructorId', 'circuitId'],
        how='left'
    )
else:
    master['constructor_podium_rate_at_circuit'] = np.nan

print("✓ Circuit-specific features created")


✓ Circuit-specific features created


## 8. Season Trends + Driver Age

Trend of position within season and driver age.

In [53]:
# Driver position trend within season (slope over last 5 races)
def calculate_season_trend(df, group_col, value_col, window=5):
    def get_slope(series):
        if len(series) < 2:
            return 0
        x = np.arange(len(series))
        return np.polyfit(x, series.values, 1)[0]

    def trend_transform(group):
        shifted = group.shift(1)
        return shifted.rolling(window=window, min_periods=2).apply(get_slope, raw=False)

    col_name = f'{value_col}_trend_season'
    df[col_name] = (
        df.groupby([group_col, 'year'])[value_col]
        .transform(trend_transform)
    )
    return df

if 'positionOrder' in master.columns:
    master = calculate_season_trend(master, 'driverId', 'positionOrder', window=5)

# Driver age
if 'dob' in master.columns and 'date' in master.columns:
    master['dob'] = pd.to_datetime(master['dob'], errors='coerce')
    master['driver_age'] = (master['date'] - master['dob']).dt.days / 365.25

print("✓ Season trend + driver age created")


✓ Season trend + driver age created


## 9. Validation Checks

Quick sanity checks to catch obvious issues.

In [54]:
def _numeric(s):
    return pd.to_numeric(s, errors='coerce')


def _count_issues(mask):
    return int(mask.sum()) if hasattr(mask, 'sum') else int(np.sum(mask))


def check_range(col, min_val=None, max_val=None, allow_na=True):
    if col not in master.columns:
        return
    s = _numeric(master[col])
    mask = pd.Series(False, index=master.index)
    if min_val is not None:
        mask |= s < min_val
    if max_val is not None:
        mask |= s > max_val
    if allow_na:
        mask &= s.notna()
    if _count_issues(mask) > 0:
        print(f"⚠ {col} out of bounds: {_count_issues(mask)}")

# Key checks
check_range('driver_standings_points', 0, 1000)
check_range('constructor_standings_points', 0, 2000)
check_range('driver_standings_position_PRE_RACE', 1, 40)
check_range('constructor_standings_position_PRE_RACE', 1, 30)
check_range('driver_total_podiums', 0, 300)
check_range('driver_races_completed', 0, 500)
check_range('driver_races_at_circuit', 0, 100)
check_range('driver_points_avg_last_10', 0, 26)

# Detailed logging for driver_points_avg_last_10 out-of-bounds
if 'driver_points_avg_last_10' in master.columns:
    s = _numeric(master['driver_points_avg_last_10'])
    oob = s.notna() & ((s < 0) | (s > 26))
    if oob.any():
        cols = [c for c in ['year', 'round', 'date', 'name', 'driverId', 'code', 'points', 'driver_points_avg_last_10'] if c in master.columns]
        print("\nOut-of-bounds driver_points_avg_last_10 rows:")
        print(master.loc[oob, cols].sort_values(['year','round','date']).head(20))

print("✓ Validation checks complete")


⚠ driver_points_avg_last_10 out of bounds: 2

Out-of-bounds driver_points_avg_last_10 rows:
      year  round       date                name  driverId code  points  \
8197  2015      4 2015-04-19  Bahrain Grand Prix         1  HAM    25.0   
8218  2015      5 2015-05-10  Spanish Grand Prix         1  HAM    18.0   

      driver_points_avg_last_10  
8197                       26.1  
8218                       26.1  
✓ Validation checks complete


## 10. Save Output

In [55]:
master.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(master):,} | Cols: {len(master.columns):,}")


Saved: data\processed\master_races_augmented.csv
Rows: 12,817 | Cols: 106


In [56]:
tmp = master[['year','round','constructorId','constructor_standings_points_PRE_RACE']].copy()
consistency = tmp.groupby(['year','round','constructorId'])['constructor_standings_points_PRE_RACE'].nunique(dropna=False)
print("Max unique PRE_RACE values within same constructor-race:", consistency.max())
print("Any constructor-race with >1 value:", (consistency > 1).sum())

Max unique PRE_RACE values within same constructor-race: 1
Any constructor-race with >1 value: 0


In [60]:
import pandas as pd
import numpy as np

cols = [
    'year', 'round', 'raceId', 'constructorId', 'driverId',
    'points',
    'constructor_standings_points',
    'constructor_standings_points_PRE_RACE'
]

df = master[cols].copy()

# Numeric safety
for c in ['points', 'constructor_standings_points', 'constructor_standings_points_PRE_RACE']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Constructor-race aggregation (so each constructor-race counted once)
cons_race = (
    df.groupby(['year', 'round', 'raceId', 'constructorId'], as_index=False)
      .agg(
          constructor_race_points=('points', 'sum'),
          constructor_standings_points=('constructor_standings_points', 'first'),
          constructor_standings_points_PRE_RACE=('constructor_standings_points_PRE_RACE', 'first')
      )
)

# Keep rows where both standings values exist
valid = cons_race.dropna(
    subset=['constructor_standings_points', 'constructor_standings_points_PRE_RACE']
).copy()

# Condition: constructor scored 0 points in race
zero_pts = valid['constructor_race_points'].eq(0)

# Expected equality when zero points scored
equal_when_zero = valid['constructor_standings_points'].eq(valid['constructor_standings_points_PRE_RACE'])

# Confirmation subset
zero_subset = valid[zero_pts].copy()
confirmed_zero = zero_subset['constructor_standings_points'].eq(
    zero_subset['constructor_standings_points_PRE_RACE']
)

n_zero = len(zero_subset)
n_confirmed = int(confirmed_zero.sum())
pct_confirmed = (n_confirmed / n_zero * 100) if n_zero else np.nan

print(f"Constructor-race rows compared: {len(valid):,}")
print(f"Zero-point constructor races: {n_zero:,}")
print(f"Confirmed (PRE_RACE == post-race standings when 0 scored): {n_confirmed:,}")
print(f"Percentage confirmed: {pct_confirmed:.2f}%" if n_zero else "Percentage confirmed: N/A (no zero-point races)")

# Optional: inspect failures
fails = zero_subset.loc[~confirmed_zero, [
    'year', 'round', 'raceId', 'constructorId',
    'constructor_race_points',
    'constructor_standings_points_PRE_RACE',
    'constructor_standings_points'
]].sort_values(['year', 'round', 'constructorId'])

print("\nSample zero-point races that did NOT match (if any):")
print(fails.head(20))

Constructor-race rows compared: 6,067
Zero-point constructor races: 2,688
Confirmed (PRE_RACE == post-race standings when 0 scored): 2,682
Percentage confirmed: 99.78%

Sample zero-point races that did NOT match (if any):
      year  round  raceId  constructorId  constructor_race_points  \
5378  2021     10    1061              9                      0.0   
5705  2022     21    1095              1                      0.0   
5712  2022     21    1095            210                      0.0   
5812  2023      9    1107            210                      0.0   
5886  2023     17    1115              3                      0.0   
6002  2024      6    1126            210                      0.0   

      constructor_standings_points_PRE_RACE  constructor_standings_points  
5378                                  286.0                         289.0  
5705                                  146.0                         148.0  
5712                                   36.0                       

In [61]:
import pandas as pd
import numpy as np

cols = [
    'year', 'round', 'raceId', 'constructorId', 'driverId',
    'points',
    'constructor_standings_points',
    'constructor_standings_points_PRE_RACE'
]

df = master[cols].copy()

# Numeric safety
for c in ['points', 'constructor_standings_points', 'constructor_standings_points_PRE_RACE']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# One row per constructor-race
cons_race = (
    df.groupby(['year', 'round', 'raceId', 'constructorId'], as_index=False)
      .agg(
          constructor_race_points=('points', 'sum'),
          constructor_standings_points=('constructor_standings_points', 'first'),
          constructor_standings_points_PRE_RACE=('constructor_standings_points_PRE_RACE', 'first')
      )
)

# Keep rows where both standings values exist
valid = cons_race.dropna(
    subset=['constructor_standings_points', 'constructor_standings_points_PRE_RACE']
).copy()

# Focus on rows where pre-race and post-race standings differ
neq = valid[
    valid['constructor_standings_points_PRE_RACE'] != valid['constructor_standings_points']
].copy()

# Expected post-race points from identity
neq['expected_post'] = neq['constructor_standings_points_PRE_RACE'] + neq['constructor_race_points']

# Float-safe comparison
neq['identity_holds'] = np.isclose(
    neq['expected_post'],
    neq['constructor_standings_points'],
    rtol=0,
    atol=1e-9
)

n = len(neq)
n_ok = int(neq['identity_holds'].sum())
pct_ok = (n_ok / n * 100) if n else np.nan

print(f"Non-equal PRE_RACE vs post-race constructor standings rows: {n:,}")
print(f"Identity holds (PRE_RACE + race_points == post-race): {n_ok:,}")
print(f"Percentage confirmed: {pct_ok:.2f}%" if n else "Percentage confirmed: N/A (no non-equal rows)")

# Optional: inspect failures
fails = neq.loc[~neq['identity_holds'], [
    'year', 'round', 'raceId', 'constructorId',
    'constructor_standings_points_PRE_RACE',
    'constructor_race_points',
    'expected_post',
    'constructor_standings_points'
]].sort_values(['year', 'round', 'constructorId'])

print("\nSample failures (if any):")
print(fails.head(20))

Non-equal PRE_RACE vs post-race constructor standings rows: 3,384
Identity holds (PRE_RACE + race_points == post-race): 3,297
Percentage confirmed: 97.43%

Sample failures (if any):
      year  round  raceId  constructorId  \
2568  2007     14      49              1   
4821  2018     13    1001             10   
5163  2020      5    1035            211   
5378  2021     10    1061              9   
5381  2021     10    1061            131   
5415  2021     14    1065              1   
5418  2021     14    1065              9   
5421  2021     14    1065            131   
5467  2021     19    1071              6   
5468  2021     19    1071              9   
5471  2021     19    1071            131   
5535  2022      4    1077              1   
5537  2022      4    1077              6   
5538  2022      4    1077              9   
5539  2022      4    1077             51   
5542  2022      4    1077            210   
5607  2022     11    1084              6   
5608  2022     11    1084 